# Apple Disease Detection — Training on Google Colab
**Track A (Yacine) — PFA 2025-2026**

This notebook runs entirely in the cloud. No local disk space needed.

### Steps
1. Install dependencies
2. Mount Google Drive (to save your trained model)
3. Upload the NVIDIA fruits images
4. Preprocess + augment
5. Train YOLOv8
6. Download the model weights

> **Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running.

---
## 1. Install Dependencies

In [ ]:
!pip install ultralytics Pillow --quiet
print('Done')

---
## 2. Mount Google Drive
Your trained model will be saved here so you don't lose it when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_PATH = '/content/drive/MyDrive/PFA/models'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f'Model will be saved to: {DRIVE_SAVE_PATH}')

---
## 3. Upload the NVIDIA Fruits Images

You have two options:

**Option A (recommended) — Upload from your computer:**
Run the cell below. It opens a file picker. Select ALL images from both folders:
- `data/raw/fruits/train/freshapples/` → all .png files
- `data/raw/fruits/train/rottenapples/` → all .png files
- `data/raw/fruits/valid/freshapples/` → all .png files
- `data/raw/fruits/valid/rottenapples/` → all .png files

**Option B — Upload zip to Drive then unzip here:**
Zip the `data/raw/fruits/` folder, upload to Google Drive, then use the unzip cell instead.

In [ ]:
# ── OPTION A: Upload files directly ──────────────────────────────────────────
from google.colab import files
import shutil, os
from pathlib import Path

# Create raw data folders
for folder in [
    '/content/raw/freshapples',
    '/content/raw/rottenapples',
]:
    os.makedirs(folder, exist_ok=True)

print('Select your images in the file picker.')
print('Hold Ctrl to select multiple files at once.')
print('Select ALL images from both freshapples and rottenapples folders.')

uploaded = files.upload()  # opens file picker

# Sort uploaded files by name prefix into the right folders
# (assumes you renamed them before uploading, OR we detect by content)
# Since we can't tell from filename which class they belong to,
# we'll ask you to upload class by class — see instructions below.
print(f'\nUploaded {len(uploaded)} files.')
print('Move them to the correct folder in the next cell.')

In [ ]:
# ── OPTION B: Unzip from Google Drive (skip if you used Option A) ─────────────
# DRIVE_ZIP = '/content/drive/MyDrive/PFA/fruits.zip'  # path to your zip in Drive
# !unzip -q "{DRIVE_ZIP}" -d /content/raw/
# print('Unzipped!')

# Skip this cell if you used Option A

### Easier Upload Method
Actually the simplest approach: **zip the fruits folder on your PC, upload the zip to Google Drive, then unzip it here.**

On your PC:
1. Right-click `data/raw/fruits/` → Send to → Compressed (zipped) folder → name it `fruits.zip`
2. Upload `fruits.zip` to Google Drive under `My Drive/PFA/`
3. Run the cell below

In [ ]:
# ── Unzip from Drive (recommended) ───────────────────────────────────────────
import os

DRIVE_ZIP = '/content/drive/MyDrive/PFA/fruits.zip'
RAW_PATH  = '/content/raw'

os.makedirs(RAW_PATH, exist_ok=True)
!unzip -q "{DRIVE_ZIP}" -d "{RAW_PATH}"

# Show what we got
!find "{RAW_PATH}" -type d
print('\nImage counts per folder:')
!find "{RAW_PATH}" -type f | awk -F/ '{print $(NF-1)}' | sort | uniq -c

---
## 4. Preprocess & Augment

In [ ]:
import random
import shutil
from pathlib import Path
from PIL import Image, ImageEnhance

# ── Config ────────────────────────────────────────────────────────────────────
RAW_ROOT   = Path('/content/raw/fruits')   # adjust if your zip extracted differently
OUT_ROOT   = Path('/content/processed')
CLASSES    = ['healthy_fruit', 'rotten_fruit']
CLASS_MAP  = {'freshapples': 'healthy_fruit', 'rottenapples': 'rotten_fruit'}
SPLIT      = (0.8, 0.1, 0.1)   # train / val / test
MIN_TRAIN  = 100                 # augment minority class up to this
SEED       = 42

# ── Collect images ────────────────────────────────────────────────────────────
all_images = {c: [] for c in CLASSES}

for split_folder in ['train', 'valid']:
    for orig_cls, our_cls in CLASS_MAP.items():
        folder = RAW_ROOT / split_folder / orig_cls
        if not folder.exists():
            print(f'[WARN] Not found: {folder}')
            continue
        imgs = list(folder.glob('*.png')) + list(folder.glob('*.jpg'))
        all_images[our_cls].extend(imgs)
        print(f'  {split_folder}/{orig_cls} → {our_cls}: {len(imgs)} images')

print()
for cls, imgs in all_images.items():
    print(f'  Total {cls}: {len(imgs)}')

In [ ]:
# ── Split + Copy ──────────────────────────────────────────────────────────────
random.seed(SEED)

def split_list(lst, train_r, val_r):
    s = lst[:]
    random.shuffle(s)
    n = len(s)
    a = int(n * train_r)
    b = int(n * val_r)
    return s[:a], s[a:a+b], s[a+b:]

def copy_imgs(imgs, dest):
    dest.mkdir(parents=True, exist_ok=True)
    for src in imgs:
        d = dest / src.name
        if d.exists():
            d = dest / f'{src.stem}_x{src.suffix}'
        shutil.copy2(src, d)

train_r, val_r, test_r = SPLIT
stats = {}

for cls in CLASSES:
    imgs = all_images[cls]
    train_imgs, val_imgs, test_imgs = split_list(imgs, train_r, val_r)
    copy_imgs(train_imgs, OUT_ROOT / 'train' / cls)
    copy_imgs(val_imgs,   OUT_ROOT / 'val'   / cls)
    copy_imgs(test_imgs,  OUT_ROOT / 'test'  / cls)
    stats[cls] = dict(train=len(train_imgs), val=len(val_imgs), test=len(test_imgs))
    print(f'  [{cls}] train={len(train_imgs)}  val={len(val_imgs)}  test={len(test_imgs)}')

In [ ]:
# ── Augment minority class ────────────────────────────────────────────────────
augmentations = [
    lambda img: img.transpose(Image.FLIP_LEFT_RIGHT),
    lambda img: img.rotate(15),
    lambda img: img.rotate(-15),
    lambda img: ImageEnhance.Brightness(img).enhance(1.3),
    lambda img: ImageEnhance.Brightness(img).enhance(0.7),
    lambda img: ImageEnhance.Contrast(img).enhance(1.4),
    lambda img: img.rotate(10),
    lambda img: img.transpose(Image.FLIP_TOP_BOTTOM),
    lambda img: ImageEnhance.Color(img).enhance(1.2),
    lambda img: img.rotate(5),
]

for cls in CLASSES:
    train_dir = OUT_ROOT / 'train' / cls
    existing  = list(train_dir.glob('*.*'))
    n_existing = len(existing)

    if n_existing >= MIN_TRAIN:
        print(f'  [{cls}] {n_existing} images — no augmentation needed')
        continue

    needed  = MIN_TRAIN - n_existing
    created = 0
    print(f'  [{cls}] {n_existing} images — generating {needed} augmented images...')

    while created < needed:
        src     = existing[created % len(existing)]
        aug_fn  = augmentations[created % len(augmentations)]
        try:
            img     = Image.open(src).convert('RGB')
            aug_img = aug_fn(img)
            out     = train_dir / f'aug_{created:04d}.jpg'
            aug_img.save(out, 'JPEG', quality=90)
            created += 1
        except Exception as e:
            print(f'    Error: {e}')

    print(f'  [{cls}] Done — now {MIN_TRAIN} train images')

print('\nFinal train counts:')
for cls in CLASSES:
    n = len(list((OUT_ROOT / 'train' / cls).glob('*.*')))
    print(f'  {cls}: {n}')

---
## 5. Train YOLOv8

In [ ]:
from ultralytics import YOLO
import torch

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
model = YOLO('yolov8n-cls.pt')  # nano classification model

results = model.train(
    data    = str(OUT_ROOT),
    epochs  = 30,          # increase to 50-100 for better accuracy
    imgsz   = 224,
    batch   = 32,
    name    = 'apple_disease_v1',
    project = '/content/runs',
    patience= 10,
    plots   = True,
)

print('\nTraining complete!')
print(f'Results saved to: {results.save_dir}')

---
## 6. Save & Download the Model

In [ ]:
from pathlib import Path
import shutil

best_pt = Path(results.save_dir) / 'weights' / 'best.pt'

# Save to Google Drive (persistent)
drive_dest = f'{DRIVE_SAVE_PATH}/apple_disease_v1_best.pt'
shutil.copy2(best_pt, drive_dest)
print(f'Saved to Drive: {drive_dest}')

# Also download to your computer
from google.colab import files
files.download(str(best_pt))
print('Downloading best.pt to your computer...')
print('\nPlace it in: PFA/ai-model/models/best.pt')

---
## 7. Quick Test

In [ ]:
# Run on the test set to see final accuracy
metrics = model.val(data=str(OUT_ROOT), split='test')
print(f'\nTest accuracy: {metrics.top1:.2%}')

In [ ]:
# Show training plots
from IPython.display import Image as IPImage
import glob

for plot in glob.glob(f'{results.save_dir}/*.png'):
    print(plot)
    display(IPImage(plot))